In [1]:
# Import the necessary libraries
import torch, json, itertools
import torch.nn.functional as F
from transformers import EsmTokenizer, EsmForMaskedLM
from tqdm import tqdm

c:\Users\catha\Documents\agentic-protein-reconstruction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the pretrained ESM-2 MLM model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # set device to cpu or gpu if available
tokeniser = EsmTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
mlm = EsmForMaskedLM.from_pretrained("facebook/esm2_t6_8M_UR50D").to(device)
mlm.eval()
print(f"Device: {device}")

Loading weights: 100%|██████████| 112/112 [00:00<00:00, 997.73it/s, Materializing param=lm_head.layer_norm.weight]                                     
EsmForMaskedLM LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     |  | 
----------------------------+------------+--+-
esm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cuda


In [3]:
# Load a sample fragmented protein
with open("../data/fragmented_ecoli.jsonl") as f:
    sample = json.loads(f.readline())
    
fragments = sample["fragments"]
original = sample["ecoli_original"]
print(f"Original Protein: {original}")
print(f"Original Length: {len(original)}")
print(f"Fragmented Protein: {fragments}")
print(f"Number of Fragments: {len(fragments)}")
print(f"Short fragments : {[(i, f) for i, f in enumerate(fragments) if len(f) <= 3]}")

Original Protein: MSQPITRENFDEWMIPVYAPAPFIPVRGEGSRLWDQQGKEYIDFAGGIAVNALGHAHPELREALNEQASKFWHTGNGYTNESVLRLAKKLIDATFADRVFFCNSGAEANEAALKLARKFAHDRYGSHKSGIVAFKNAFHGRTLFTVSAGGQPAYSQDFAPLPPDIRHAAYNDINSASALIDDSTCAVIVEPIQGEGGVVPASNAFLQGLRELCDRHNALLIFDEVQTGVGRTGELYAYMHYGVTPDLLTTAKALGGGFPVGALLATEECASVMTVGTHGTTYGGNPLASAVAGKVLELINTPEMLNGVKQRHDWFVERLNIINHRYGLFNEVRGLGLLIGCVLNADYAGQAKQISQEAAKAGVMVLIAGGNVVRFAPALNVSEEEVTTGLDRFAVACEHFVSRGSS
Original Length: 406
Fragmented Protein: ['GSS', 'LAR', 'EYIDFAGGIAVNALGHAHPELR', 'NAFHGR', 'FWHTGNGYTNESVLR', 'K', 'HNALLIFDEVQTGVGR', 'LWDQQGK', 'QISQEAAK', 'FAHDR', 'ENFDEWMIPVYAPAPFIPVR', 'HAAYNDINSASALIDDSTCAVIVEPIQGEGGVVPASNAFLQGLR', 'SGIVAFK', 'FAVACEHFVSR', 'VFFCNSGAEANEAALK', 'FAPALNVSEEEVTTGLDR', 'GEGSR', 'TLFTVSAGGQPAYSQDFAPLPPDIR', 'LIDATFADR', 'VLELINTPEMLNGVK', 'EALNEQASK', 'ELCDR', 'YGLFNEVR', 'YGSHK', 'AGVMVLIAGGNVVR', 'K', 'HDWFVER', 'LNIINHR', 'TGELYAYMHYGVTPDLLTTAK', 'GLGLLIGCVLNADYAGQAK', 'ALGGGFPVGALLATEECASVMTVGTHGTTYGGNPLASAVAGK', 'MSQPITR', 'QR'

In [4]:
def score_junctions(fragments, batch_size=32):
    """Score all pairwise fragment junctions."""
    n = len(fragments)
    pairs = list(itertools.permutations(range(n), 2))
    inputs, targets = [], []

    for i, j in pairs:
        a, b = fragments[i], fragments[j]
        if len(b) == 1: # b is a single residue, mask first of a
            masked = a[:-1] + tokeniser.mask_token + b
            targets.append([a[-1]])
        elif len(a) == 1: # a is a single residue, mask first of b
            masked = a + tokeniser.mask_token + b[1:]
            targets.append([b[0]])
        else: # mask last of a and first of b
            masked = a[:-1] + tokeniser.mask_token + tokeniser.mask_token + b[1:]
            targets.append([a[-1], b[0]])
        inputs.append(masked)
    mat = torch.zeros(n, n)

    for start in tqdm(range(0, len(pairs), batch_size), desc="Scoring Junctions"):
        end = min(start + batch_size, len(pairs))
        batch = tokeniser(
            inputs[start:end],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        )
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.no_grad():
            logits = mlm(**batch).logits

        for k, (i, j) in enumerate(pairs[start:end]):
            mask_pos = (
                batch["input_ids"][k] == tokeniser.mask_token_id
            ).nonzero(as_tuple=False)[:, 0]

            score = sum(
                F.log_softmax(logits[k, m], dim=-1)[
                    tokeniser.convert_tokens_to_ids(aa)
                ].item()
                for m, aa in zip(mask_pos, targets[start + k])
            )

            mat[i, j] = score / len(targets[start + k])

    return mat

def greedy_order(scores):
    """Greedily pick the next fragment with highest score from current fragment."""
    n = scores.shape[0]
    used = set()
    start = scores.sum(dim=0).argmin().item()
    order = [start]
    used.add(start)
    while len(order) < n:
        row = scores[order[-1]].clone()
        row[list(used)] = float("-inf")
        nxt = row.argmax().item()
        order.append(nxt)
        used.add(nxt)
    return order

In [5]:
# Reconstruct the sample
scores = score_junctions(fragments) # score the junctions
order = greedy_order(scores) # greedy ordering
fragmented = "".join(fragments)
result = "".join(fragments[i] for i in order)
print(f"Original:      {original}")
print(f"Fragmented:    {fragmented}")
print(f"Reconstructed: {result}")

Scoring Junctions: 100%|██████████| 36/36 [00:01<00:00, 28.19it/s]

Original:      MSQPITRENFDEWMIPVYAPAPFIPVRGEGSRLWDQQGKEYIDFAGGIAVNALGHAHPELREALNEQASKFWHTGNGYTNESVLRLAKKLIDATFADRVFFCNSGAEANEAALKLARKFAHDRYGSHKSGIVAFKNAFHGRTLFTVSAGGQPAYSQDFAPLPPDIRHAAYNDINSASALIDDSTCAVIVEPIQGEGGVVPASNAFLQGLRELCDRHNALLIFDEVQTGVGRTGELYAYMHYGVTPDLLTTAKALGGGFPVGALLATEECASVMTVGTHGTTYGGNPLASAVAGKVLELINTPEMLNGVKQRHDWFVERLNIINHRYGLFNEVRGLGLLIGCVLNADYAGQAKQISQEAAKAGVMVLIAGGNVVRFAPALNVSEEEVTTGLDRFAVACEHFVSRGSS
Fragmented:    GSSLAREYIDFAGGIAVNALGHAHPELRNAFHGRFWHTGNGYTNESVLRKHNALLIFDEVQTGVGRLWDQQGKQISQEAAKFAHDRENFDEWMIPVYAPAPFIPVRHAAYNDINSASALIDDSTCAVIVEPIQGEGGVVPASNAFLQGLRSGIVAFKFAVACEHFVSRVFFCNSGAEANEAALKFAPALNVSEEEVTTGLDRGEGSRTLFTVSAGGQPAYSQDFAPLPPDIRLIDATFADRVLELINTPEMLNGVKEALNEQASKELCDRYGLFNEVRYGSHKAGVMVLIAGGNVVRKHDWFVERLNIINHRTGELYAYMHYGVTPDLLTTAKGLGLLIGCVLNADYAGQAKALGGGFPVGALLATEECASVMTVGTHGTTYGGNPLASAVAGKMSQPITRQRLAK
Reconstructed: HAAYNDINSASALIDDSTCAVIVEPIQGEGGVVPASNAFLQGLRELCDRHNALLIFDEVQTGVGRLNIINHRLARGEGSRGSSLAKVLELINTPEMLNGVKVFFCNSGAEANEAALKALGGGFPVGALLATEECASVMTVG